# Dynamics of VaR and CVaR

In [64]:
import os
from typing import Tuple, List
import numpy as np
import pandas as pd
import plotly.express as px
pd.options.plotting.backend = "plotly"

import sys
sys.path.append(os.getcwd().split("scripts")[0])
sys.path.append(os.path.join(os.getcwd().split("scripts")[0], "params"))

from tqdm import tqdm
from params import funding, caps
import pystable

## Loading Data

In [65]:
filename = "btc"

path_to_file = os.path.join(os.getcwd().split("scripts")[0], f"data/{filename}")

periodicity = 24 * 60 * 60. # 1 day in seconds

In [66]:
def compute_log_return(_df:pd.DataFrame, column:str) -> pd.DataFrame:
    tmp_df = _df.copy()
    if not column in tmp_df.columns:
        raise Exception(f"Column {column} not found")
    tmp_df["log_return"] = np.nan
    tmp_df.loc[tmp_df.index[1]:,"log_return"] = np.log(
        tmp_df[column].iloc[1:].to_numpy() / tmp_df[column].iloc[:-1].to_numpy()
    )
    tmp_df.dropna(inplace=True)
    return tmp_df

In [67]:
df = pd.read_csv(path_to_file+".csv", parse_dates=["timestamp"]).set_index("timestamp")
df = compute_log_return(df, "close")
df

,close,log_return
timestamp,,
2023-01-01 00:02:00+00:00,16539.31,-2.635796e-04
2023-01-01 00:03:00+00:00,16536.43,-1.741458e-04
2023-01-01 00:04:00+00:00,16533.65,-1.681278e-04
2023-01-01 00:05:00+00:00,16535.38,1.046296e-04
2023-01-01 00:06:00+00:00,16536.70,7.982564e-05
...,...,...
2024-05-14 19:12:00+00:00,61497.14,6.041130e-04
2024-05-14 19:13:00+00:00,61486.37,-1.751454e-04
2024-05-14 19:14:00+00:00,61462.15,-3.939860e-04


In [68]:
df_daily = df[["close"]].resample("1D", label="right").last()
df_daily = compute_log_return(df_daily, "close")
df_daily

,close,log_return
timestamp,,
2023-01-03 00:00:00+00:00,16675.15,0.003532
2023-01-04 00:00:00+00:00,16674.48,-0.000040
2023-01-05 00:00:00+00:00,16853.11,0.010656
2023-01-06 00:00:00+00:00,16832.12,-0.001246
2023-01-07 00:00:00+00:00,16950.43,0.007004
...,...,...
2024-05-11 00:00:00+00:00,60804.00,-0.036336
2024-05-12 00:00:00+00:00,60809.99,0.000099
2024-05-13 00:00:00+00:00,61476.09,0.010894


The lowest return is different.

In [69]:
df_lowest = df[["close"]].resample("1D", label="right").min()
tmp_df_daily = df[["close"]].resample("1D", label="right").last()
#df_lowest = compute_log_return(df_lowest, "close")

df_lowest["log_return"] = np.nan
df_lowest.loc[df_lowest.index[1]:,"log_return"] = np.log(
    df_lowest["close"].iloc[1:].to_numpy() / tmp_df_daily["close"].iloc[:-1].to_numpy()
)
df_lowest.dropna(inplace=True)

df_lowest

,close,log_return
timestamp,,
2023-01-03 00:00:00+00:00,16552.74,-0.003836
2023-01-04 00:00:00+00:00,16608.25,-0.004020
2023-01-05 00:00:00+00:00,16656.36,-0.001087
2023-01-06 00:00:00+00:00,16774.11,-0.004699
2023-01-07 00:00:00+00:00,16688.03,-0.008597
...,...,...
2024-05-11 00:00:00+00:00,60236.93,-0.045706
2024-05-12 00:00:00+00:00,60530.91,-0.004501
2024-05-13 00:00:00+00:00,60654.01,-0.002568


In [70]:
def fit_distribution(
    _df:pd.DataFrame, column:str="log_return", period:float=periodicity, verbose:bool=False
) -> Tuple[pystable.STABLE_DIST, pystable.STABLE_DIST]:

    if not column in _df.columns:
        raise Exception(f"Column {column} not found")

    _orig_dst = funding.gaussian()
    pystable.fit(_orig_dst, _df[column].to_numpy(), _df.index.size)

    if verbose:
        print(f'''
            original fitted distribution
            alpha: {_orig_dst.contents.alpha}, beta: {_orig_dst.contents.beta},
            mu: {_orig_dst.contents.mu_1}, sigma: {_orig_dst.contents.sigma}
            '''
        )

    # this rescale changes from the original unit of the data to seconds
    _rescaled_dst = caps.rescale(_orig_dst, 1./period)
    _rescaled_dst_2 = funding.rescale(_orig_dst, 1./period)

    if verbose:
        print(f'''
            (caps.py) rescaled distribution (1/t = {1./period}):
            alpha: {_rescaled_dst.contents.alpha}, beta: {_rescaled_dst.contents.beta},
            mu: {_rescaled_dst.contents.mu_1}, sigma: {_rescaled_dst.contents.sigma}
            '''
        )
        print(f'''
            (funding.py) rescaled distribution (1/t = {1./period}):
            alpha: {_rescaled_dst_2.contents.alpha}, beta: {_rescaled_dst_2.contents.beta},
            mu: {_rescaled_dst_2.contents.mu_1}, sigma: {_rescaled_dst_2.contents.sigma}
            '''
        )

    return _orig_dst, _rescaled_dst

In [71]:
orig_dst, rescaled_dst = fit_distribution(
    df_daily, column="log_return", period=periodicity, verbose=True
)


            original fitted distribution
            alpha: 1.356988564065287, beta: 0.17304813534895266,
            mu: 0.0030127160573657, sigma: 0.011481286475040433
            

            (caps.py) rescaled distribution (1/t = 1.1574074074074073e-05):
            alpha: 1.356988564065287, beta: 0.17304813534895266,
            mu: 3.486939881210301e-08, sigma: 3.310126674692034e-06
            

            (funding.py) rescaled distribution (1/t = 1.1574074074074073e-05):
            alpha: 1.356988564065287, beta: 0.17304813534895266,
            mu: 3.486939881210301e-08, sigma: 2.6432959433373516e-06
            


## Compute VaR from Distribution

In [72]:
def VaR_from_dist(_any_dst:pystable.STABLE_DIST, _forecast:int, _alphas: np.ndarray) -> np.ndarray:
    """
    Compute F^{-1}_{X_t}(1-alpha) using pystable
    * _any_dst: any distribution
    * _forecast: the forecast period (in the unit of the distribution)
    * _alphas: the 
    """

    # this rescale changes from seconds to "_forecast"
    _dst_y = caps.rescale(_any_dst, _forecast)

    return np.array(pystable.q(_dst_y, _alphas, len(_alphas)))

In [73]:
funding.ALPHAS

array([0.01 , 0.025, 0.05 , 0.075, 0.1  ])

In [74]:
one_day_var_from_dist = VaR_from_dist(orig_dst, 1, funding.ALPHAS)
one_day_var_from_dist

array([-0.10589606, -0.05608003, -0.03564908, -0.02755106, -0.02284126])

In [75]:
df_lowest.loc[df_lowest.index[-1], "log_return"]

-0.028027035449195728

In [76]:
one_day_var_from_scaled_dist = VaR_from_dist(rescaled_dst, periodicity, funding.ALPHAS)
one_day_var_from_scaled_dist

array([-0.10589606, -0.05608003, -0.03564908, -0.02755106, -0.02284126])

### Backtest VaR from Distribution - Coverage

In [77]:
def backtest_var_from_dist(
    _df:pd.DataFrame, _lookback_window:int, _forecast:int
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    df_var_full = pd.DataFrame(
        0.,
        columns=[f"{alpha:.3f}" for alpha in funding.ALPHAS],
        index=pd.date_range(
            start=_df.index[0].floor("D")+pd.Timedelta(days=_lookback_window), end=_df.index[-2].floor("D"), freq='1D'
        )
    )

    df_var_partial = df_var_full.copy()

    pbar = tqdm(total=df_var_full.index.size)

    for idx in df_var_full.index:

        # --------------
        # full past data
        # --------------

        orig_dst, _ = fit_distribution(
            df_daily.loc[:idx,:], "log_return", periodicity, verbose=False
        )

        df_var_full.loc[idx, :] = VaR_from_dist(orig_dst, _forecast, funding.ALPHAS)

        # -----------------
        # partial past data - 1 month lookback window
        # -----------------

        past_idx = idx - pd.Timedelta(days=_lookback_window)

        partial_orig_dst, _ = fit_distribution(
            df_daily.loc[past_idx:idx,:], "log_return", periodicity, verbose=False
        )

        df_var_partial.loc[idx, :] = VaR_from_dist(partial_orig_dst, _forecast, funding.ALPHAS)

        pbar.update(1)

    pbar.close()

    return df_var_full, df_var_partial


In [78]:
lookback_window = 60  # days
forecast = 1  # day

In [79]:
df_var_full, df_var_partial = backtest_var_from_dist(
    df_daily, lookback_window, forecast
)

start_idx, last_idx = df_var_partial.index[1], df_var_partial.index[-1]
lowest = df_lowest.loc[start_idx:, "log_return"].to_numpy()

df_var_full.loc[:last_idx, "lowest"] = lowest
df_var_partial.loc[:last_idx, "lowest"] = lowest


100%|██████████| 438/438 [00:02<00:00, 183.91it/s]


In [80]:
def plot_coverage(_df:pd.DataFrame, _alpha:str, metric:str="VaR") -> None:

    if not _alpha in _df.columns:
        raise Exception(f"Column {_alpha} not found")

    coverage = 100. * (_df["lowest"] <= _df[_alpha]).sum() / float(_df.index.size)

    confidence = 100. * (1-float(_alpha))

    return _df[["lowest", _alpha]].plot(
        title=f"1-day {metric} coverage at {confidence:.1f}% confidence: {coverage:.3f}%"
    )

### VaR Results with full past data

In [81]:
plot_coverage(df_var_full, "0.100")

In [82]:
plot_coverage(df_var_full, "0.050")

In [83]:
plot_coverage(df_var_full, "0.025")

In [84]:
plot_coverage(df_var_full, "0.010")

### VaR Results with partial past data

Lookback window of 30 days

In [85]:
plot_coverage(df_var_partial, "0.100")

In [86]:
plot_coverage(df_var_partial, "0.050")

In [87]:
plot_coverage(df_var_partial, "0.025")

In [88]:
plot_coverage(df_var_partial, "0.010")

## Computing CVaR from Data

In [89]:
def CVaR_from_data(_s:pd.Series, var:np.ndarray) -> np.ndarray:

    cvar = np.zeros_like(var)

    for i, v in enumerate(var):
        _tmp_s:pd.Series = _s.loc[_s <= v]
        cvar[i] = _tmp_s.mean()

    return cvar

In [90]:
one_day_cvar_from_data = CVaR_from_data(df_lowest["log_return"], one_day_var_from_dist)
one_day_var_from_dist, one_day_cvar_from_data

(array([-0.10589606, -0.05608003, -0.03564908, -0.02755106, -0.02284126]),
 array([-0.13061887, -0.07830868, -0.05648422, -0.04936836, -0.04378526]))

### Backtesting CVaR from Data Coverage

In [91]:
def backtest_cvar_from_data(
    _df:pd.DataFrame, _df_var_full:pd.DataFrame, _df_var_partial:pd.DataFrame, _lookback_window:int
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    _df_cvar_full = _df_var_full.copy()
    _df_cvar_partial = _df_var_partial.copy()

    alphas = [f"{alpha:.3f}" for alpha in funding.ALPHAS]

    pbar = tqdm(total=_df_cvar_partial.index.size)

    for idx in _df_cvar_full.index:

        _df_cvar_full.loc[idx,alphas] = CVaR_from_data(
            _df.loc[:idx,"log_return"], _df_var_full.loc[idx,alphas].to_numpy()
        )

        # -----------------
        # partial past data - 1 month lookback window
        # -----------------

        past_idx = idx - pd.Timedelta(days=_lookback_window)

        _df_cvar_partial.loc[idx,alphas] = CVaR_from_data(
            _df.loc[past_idx:idx,"log_return"], _df_var_partial.loc[idx,alphas].to_numpy()
        )

        pbar.update(1)

    pbar.close()

    return _df_cvar_full, _df_cvar_partial

In [92]:
df_cvar_full, df_cvar_partial = backtest_cvar_from_data(
    df_daily, df_var_full, df_var_partial, lookback_window
)

100%|██████████| 438/438 [00:00<00:00, 503.90it/s]


### CVaR Results with full past data

In [93]:
plot_coverage(df_cvar_full, "0.100", metric="CVaR")

In [94]:
plot_coverage(df_cvar_full, "0.050", metric="CVaR")

In [95]:
plot_coverage(df_cvar_full, "0.025", metric="CVaR")

In [96]:
plot_coverage(df_cvar_full, "0.010", metric="CVaR")

### CVaR Results with partial past data

In [97]:
plot_coverage(df_cvar_partial, "0.100", metric="CVaR")

In [98]:
plot_coverage(df_cvar_partial, "0.050", metric="CVaR")

In [99]:
plot_coverage(df_cvar_partial, "0.025", metric="CVaR")

In [100]:
plot_coverage(df_cvar_partial, "0.010", metric="CVaR")

## Computing CVaR from Distribution

In [116]:
def CVaR_from_dist(
    any_dst:pystable.STABLE_DIST, _forecast:int, alphas:np.ndarray
) -> np.ndarray:

    cvars:np.ndarray = np.zeros_like(alphas)
    _dst_y:pystable.STABLE_DIST = caps.rescale(any_dst, _forecast)
    min_var = pystable.q(_dst_y, [0.001 * np.min(alphas)], 1)[0]
    fixed_interval = 100

    for i, a in enumerate(alphas):
        var:np.ndarray = pystable.q(_dst_y, [a], 1)[0]
        x = np.linspace(min_var, var, fixed_interval)
        prob = np.array(pystable.pdf(_dst_y, x, len(x)))
        cvars[i] = np.dot(x, prob) / sum(prob)  # conditional probability

    return cvars

In [118]:
CVaR_from_dist(orig_dst, 1, funding.ALPHAS)

array([-0.18400445, -0.07724183, -0.04289368, -0.0314886 , -0.02548088])

In [119]:
one_day_cvar_from_data

array([-0.13061887, -0.07830868, -0.05648422, -0.04936836, -0.04378526])

In [114]:
one_day_var_from_dist

array([-0.10589606, -0.05608003, -0.03564908, -0.02755106, -0.02284126])

### Backtest CVaR from Distribution - Coverage

In [121]:
def backtest_cvar_from_dist(
    _df:pd.DataFrame, _lookback_window:int, _forecast:int
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    alphas = [f"{alpha:.3f}" for alpha in funding.ALPHAS]

    _df_cvar_full = pd.DataFrame(
        0.,
        columns=alphas,
        index=pd.date_range(
            start=_df.index[0].floor("D")+pd.Timedelta(days=_lookback_window), end=_df.index[-2].floor("D"), freq='1D'
        )
    )

    _df_cvar_partial = _df_cvar_full.copy()

    pbar = tqdm(total=_df_cvar_full.index.size)

    for idx in _df_cvar_full.index:

        # --------------
        # full past data
        # --------------

        orig_dst, _ = fit_distribution(
            _df.loc[:idx,:], "log_return", periodicity, verbose=False
        )

        _df_cvar_full.loc[idx, :] = CVaR_from_dist(orig_dst, _forecast, funding.ALPHAS)

        # -----------------
        # partial past data - lookback window
        # -----------------

        past_idx = idx - pd.Timedelta(days=_lookback_window)

        partial_orig_dst, _ = fit_distribution(
            _df.loc[past_idx:idx,:], "log_return", periodicity, verbose=False
        )

        _df_cvar_partial.loc[idx, :] = CVaR_from_dist(partial_orig_dst, _forecast, funding.ALPHAS)

        pbar.update(1)

    pbar.close()

    return _df_cvar_full, _df_cvar_partial


In [122]:
df_cvar_from_dist_full, df_cvar_from_dist_partial = backtest_cvar_from_dist(
    df_daily, lookback_window, forecast
)

start_idx, last_idx = df_cvar_from_dist_partial.index[1], df_cvar_from_dist_partial.index[-1]
lowest = df_lowest.loc[start_idx:, "log_return"].to_numpy()

df_cvar_from_dist_full.loc[:last_idx, "lowest"] = lowest
df_cvar_from_dist_partial.loc[:last_idx, "lowest"] = lowest

100%|██████████| 438/438 [00:04<00:00, 97.32it/s]


### CVaR Results with full past data

In [129]:
plot_coverage(df_cvar_from_dist_full, _alpha="0.100", metric="CVaR from dist")

In [125]:
plot_coverage(df_cvar_from_dist_full, _alpha="0.075", metric="CVaR from dist")

In [126]:
plot_coverage(df_cvar_from_dist_full, _alpha="0.050", metric="CVaR from dist")

In [127]:
plot_coverage(df_cvar_from_dist_full, _alpha="0.025", metric="CVaR from dist")

In [128]:
plot_coverage(df_cvar_from_dist_full, _alpha="0.010", metric="CVaR from dist")

### CVaR Results with partial past data

In [132]:
plot_coverage(df_cvar_from_dist_partial, _alpha="0.100", metric="Partial CVaR from dist")

In [133]:
plot_coverage(df_cvar_from_dist_partial, _alpha="0.075", metric="Partial CVaR from dist")

In [134]:
plot_coverage(df_cvar_from_dist_partial, _alpha="0.050", metric="Partial CVaR from dist")

In [135]:
plot_coverage(df_cvar_from_dist_partial, _alpha="0.025", metric="Partial CVaR from dist")

In [136]:
plot_coverage(df_cvar_from_dist_partial, _alpha="0.010", metric="Partial CVaR from dist")